# Processamento e Transformação da Camada Silver (`02_transformacao_silver.py`)

Este notebook realiza a limpeza, padronização, enriquecimento e garantia de qualidade das tabelas da **Camada Bronze** do ecossistema da Copa do Mundo 2026.

### Fluxo de Execução Otimizado:
1. **Configuração e Módulos Auxiliares:** Criação de rotinas reutilizáveis para conversão em `snake_case` e restauração de codificação de texto (*Mojibake*).
2. **Transformação Individual por Tabela (Limpeza, Deduplicação e Filtragem):** Aplicação de tratamentos primários diretamente nas tabelas Bronze para garantir chaves limpas e remover atributos desnecessários.
3. **Enriquecimento Cruzado de Dados (*Data Enrichment*):** Junção e mapeamento de confederações, etapas de torneio e contagens agregadas sobre dados já deduplicados.
4. **Validação de Qualidade de Dados (*Data Quality Check*):** Verificação de consistência (ex.: validação de posse de bola < 100% em cada partida).
5. **Persistência em Delta Lake no Unity Catalog:** Gravação das 6 tabelas no schema `copa_do_mundo.silver`.

In [0]:
import re
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, regexp_replace, sum, when

# Definição de catálogo e schemas no Unity Catalog
CATALOG_NAME = "copa_do_mundo"
SILVER_SCHEMA = "silver"
BRONZE_SCHEMA = "bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")


# 1. Função Auxiliar: Padronização de colunas para snake_case
def to_snake_case(df: DataFrame) -> DataFrame:
    for old_col in df.columns:
        new_col = old_col.strip()
        new_col = re.sub(r"[^\w\s]", "", new_col)  # Remove símbolos e pontuações
        new_col = re.sub(r"[\s-]+", "_", new_col)  # Converte espaços e hífens para _
        new_col = new_col.lower()
        df = df.withColumnRenamed(old_col, new_col)
    return df


# 2. Nova Função Auxiliar: Substituição direta e determinística de Mojibake
def fix_mojibake(df: DataFrame) -> DataFrame:
    # Dicionário completo de de-para dos artefatos identificados nos dados
    mojibake_map = {
        "TÃ¼rkiye": "Türkiye",
        "Ã¼": "ü",
        "Ã§": "ç",
        "Ã´": "ô",
        "Ã©": "é",
        "Ã£": "ã",
        "Ã­": "í",
        "Ã¥": "å",
        "Ã³": "ó",
        "Ãº": "ú",
        "Ä": "č",
        "Ä‡": "ć",
        "Ã¡": "á",
        "Ã«": "ë",
        "Ã": "Á",
        "ÅŸ": "ş",
        "Å¡": "š",
        "Ã¯": "ï",
        "Ã¨": "è",
        "Ã±": "ñ",
        "Ã‡": "Ç",
        "ÄŸ": "ğ",
        "ÃŸ": "ß",
        "Ã¶": "ö",
        "Ã˜": "Ø",
        "Ã¸": "ø",
        "Ã¦": "æ",
        "Ä›": "ě",
        "Ä°": "İ",
        "Ä±": "ı",
        "Å™": "ř",
        "â€™": "'",
    }

    # Identifica colunas do tipo String no DataFrame
    string_cols = [
        field.name
        for field in df.schema.fields
        if field.dataType.typeName() == "string"
    ]

    # Aplica as substituições em todas as colunas do tipo texto
    for c in string_cols:
        for mojibake, clean in mojibake_map.items():
            df = df.withColumn(c, regexp_replace(col(c), mojibake, clean))

    return df

## Passo 1: Leitura, Padronização, Limpeza de Encoding e Deduplicação Individual
Nesta etapa, processamos cada tabela da Bronze isoladamente para garantir que as chaves primárias estejam limpas e únicas antes dos *joins*.

In [0]:
# --- 1. MATCHES_DETAILED ---
df_matches = spark.read.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.matches_detailed_bronze"
)
df_matches = to_snake_case(df_matches)
df_matches = df_matches.dropDuplicates(["match_id"])

# --- 2. MATCH_TEAM_STATS ---
df_team_stats = spark.read.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.match_team_stats_bronze"
)
df_team_stats = to_snake_case(df_team_stats)
df_team_stats = fix_mojibake(df_team_stats)
# Nota: NÃO aplicamos dropDuplicates em match_id pois existem 2 registros por partida (1 por seleção)

# --- 3. SQUADS_AND_PLAYERS ---
df_squads = spark.read.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.squads_and_players_bronze"
)
df_squads = to_snake_case(df_squads)
df_squads = fix_mojibake(df_squads)
df_squads = df_squads.dropDuplicates(["player_id"])

# --- 4. PLAYER_STATS ---
df_player_stats = spark.read.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.player_stats_bronze"
)
df_player_stats = to_snake_case(df_player_stats)
df_player_stats = fix_mojibake(df_player_stats)
df_player_stats = df_player_stats.dropDuplicates(["player_id"])

# --- 5. REFEREES ---
df_referees = spark.read.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.referees_bronze"
)
df_referees = to_snake_case(df_referees)
df_referees = fix_mojibake(df_referees)
df_referees = df_referees.dropDuplicates(["referee_id"])

# --- 6. MATCH_PREDICTION_FEATURES ---
df_predictions = spark.read.table(
    f"{CATALOG_NAME}.{BRONZE_SCHEMA}.match_prediction_features_bronze"
)
df_predictions = to_snake_case(df_predictions)
df_predictions = fix_mojibake(df_predictions)
df_predictions = df_predictions.dropDuplicates(["match_id"])

# Remoção de colunas que não serão utilizadas
cols_to_drop = [
    "home_squad_total_caps",
    "away_squad_total_caps",
    "home_squad_total_value_eur",
    "away_squad_total_value_eur",
    "home_squad_avg_value_eur",
    "away_squad_avg_value_eur",
    "home_rest_days",
    "away_rest_days",
    "home_prev_avg_goals_scored",
    "home_prev_avg_goals_conceded",
    "away_prev_avg_goals_scored",
    "away_prev_avg_goals_conceded",
    "home_prev_avg_possession",
    "away_prev_avg_possession",
    "home_prev_avg_shots",
    "away_prev_avg_shots",
    "home_prev_avg_shots_on_target",
    "away_prev_avg_shots_on_target",
    "home_prev_avg_saves",
    "away_prev_avg_saves",
    "home_prev_avg_corners",
    "away_prev_avg_corners",
    "home_prev_avg_fouls",
    "away_prev_avg_fouls",
    "home_prev_avg_offsides",
    "away_prev_avg_offsides",
    "home_prev_avg_xg_scored",
    "home_prev_avg_xg_conceded",
    "away_prev_avg_xg_scored",
    "away_prev_avg_xg_conceded",
]

df_predictions = df_predictions.drop(*cols_to_drop)

## Passo 2: Enriquecimento Cruzado de Dados (*Data Enrichment*)

Com as tabelas limpas e sem duplicatas, executamos os cruzamentos e derivações de colunas:
1. **Mapeamento de Confederações:** Atribuição da confederação internacional aos árbitros com base na nacionalidade.
2. **Métricas Agregadas por Árbitro:** Cálculo do total de partidas apitadas e total de faltas marcadas.
3. **Mapeamento de Estágio do Torneio:** Tradução de `stage_id` para `stage_name` na tabela `match_prediction_features`.
4. **Vínculo Árbitro-Confronto:** Inclusão da confederação do árbitro na tabela `match_prediction_features`.

In [0]:
# A. Mapeamento da Confederação na tabela de Árbitros
df_referees = df_referees.withColumn(
    "confederacao",
    when(
        col("country").isin(
            "Argentina",
            "Brazil",
            "Chile",
            "Colombia",
            "Ecuador",
            "Paraguay",
            "Peru",
            "Uruguay",
            "Venezuela",
            "Bolivia",
        ),
        "CONMEBOL",
    )
    .when(
        col("country").isin(
            "Germany",
            "France",
            "England",
            "Spain",
            "Italy",
            "Portugal",
            "Netherlands",
            "Belgium",
            "Croatia",
            "Slovenia",
            "Poland",
            "Slovakia",
            "Norway",
            "Sweden",
            "Romania",
        ),
        "UEFA",
    )
    .when(
        col("country").isin("USA", "Mexico", "Canada", "Costa Rica", "Jamaica", "El Salvador", "Honduras"),
        "CONCACAF",
    )
    .when(
        col("country").isin(
            "Japan", "South Korea", "Australia", "Saudi Arabia", "Qatar", "Iran", "China", "Jordan"
        ),
        "AFC",
    )
    .when(
        col("country").isin(
            "Senegal", "Morocco", "Nigeria", "Egypt", "Cameroon", "Algeria", "South Africa"
        ),
        "CAF",
    )
    .otherwise("OFC / Outros"),
)

# B. Métricas agregadas por árbitro (Partidas e Faltas)
df_referee_matches = df_predictions.groupBy("referee_id").agg(
    count("match_id").alias("total_jogos_apitados")
)

df_referee_fouls = (
    df_predictions.select("match_id", "referee_id")
    .join(
        df_team_stats.select("match_id", "fouls"),
        on="match_id",
        how="inner",
    )
    .groupBy("referee_id")
    .agg(sum("fouls").alias("total_faltas_marcadas"))
)

# Enriquecendo a tabela de árbitros
df_referees = (
    df_referees.join(df_referee_matches, on="referee_id", how="left")
    .join(df_referee_fouls, on="referee_id", how="left")
    .fillna(0, subset=["total_jogos_apitados", "total_faltas_marcadas"])
)

# C. Enriquecimento da tabela match_prediction_features
# 1. Mapeamento de stage_id para stage_name a partir de matches_detailed
df_stage_mapping = df_matches.select("match_id", "stage_name").distinct()
df_predictions = df_predictions.join(
    df_stage_mapping, on="match_id", how="left"
)

# 2. Mapeamento da confederação do árbitro na partida
df_referee_conf = df_referees.select(
    col("referee_id"), col("confederacao").alias("referee_confederacao")
)
df_predictions = df_predictions.join(
    df_referee_conf, on="referee_id", how="left"
)

# D. Enriquecimento da tabela player_stats com date_of_birth
df_player_dob = df_squads.select("player_id", "date_of_birth").distinct()
df_player_stats = df_player_stats.join(
    df_player_dob, on="player_id", how="left"
)

## Passo 3: Validação de Qualidade de Dados (*Data Quality Check*)

Verificação da regra de negócio na tabela `match_team_stats`: a soma da posse de bola (`possession_pct`) de ambas as seleções na mesma partida não deve exceder **100%**.

In [0]:
# Agrupamento da posse por partida
df_possession_check = df_team_stats.groupBy("match_id").agg(
    sum("possession_pct").alias("total_possession")
)

# Verificação de inconsistências (> 100%)
invalid_possession_matches = df_possession_check.filter(
    col("total_possession") > 100
)

invalid_count = invalid_possession_matches.count()

if invalid_count > 0:
    print(
        f"⚠️ ALERTA DE QUALIDADE: Foram encontradas {invalid_count} partidas com soma de posse de bola superior a 100%!"
    )
    invalid_possession_matches.show()
else:
    print(
        "✅ VALIDAÇÃO CONCLUÍDA: Nenhuma partida excede 100% de posse de bola."
    )

## Passo 4: Persistência das Tabelas Delta Lake no Unity Catalog

Gravação das 6 tabelas tratadas no catálogo `copa_do_mundo.silver`.

In [0]:
tables_to_save = {
    "matches_detailed_silver": df_matches,
    "match_team_stats_silver": df_team_stats,
    "squads_and_players_silver": df_squads,
    "player_stats_silver": df_player_stats,
    "referees_silver": df_referees,
    "match_prediction_features_silver": df_predictions,
}

for table_name, df_table in tables_to_save.items():
    full_table_path = f"{CATALOG_NAME}.{SILVER_SCHEMA}.{table_name}"
    print(f"Salvando tabela: {full_table_path}...")

    df_table.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(full_table_path)

print("🚀 Todas as tabelas da Camada Silver foram processadas e salvas com sucesso!")

## Passo 5: Validação das transformações na Camada Silver

Verificação 6 tabelas tratadas no catálogo `copa_do_mundo.silver`.

In [0]:
# Lista com o nome de todas as tabelas criadas na Camada Silver
silver_tables = [
    "matches_detailed_silver",
    "match_team_stats_silver",
    "squads_and_players_silver",
    "player_stats_silver",
    "referees_silver",
    "match_prediction_features_silver",
]

CATALOG_NAME = "copa_do_mundo"
SILVER_SCHEMA = "silver"

for table in silver_tables:
    full_table_path = f"{CATALOG_NAME}.{SILVER_SCHEMA}.{table}"

    print("=" * 80)
    print(f"📊 TABELA: {full_table_path}")
    print("=" * 80)

    # Lendo a tabela Delta do Unity Catalog
    df_temp = spark.read.table(full_table_path)

    # Exibindo contagem total de registros
    print(f"Total de registros: {df_temp.count()}\n")

    # Exibindo as 10 primeiras linhas no formato interativo do Databricks
    display(df_temp.limit(10))